In [ ]:

from CART.analyzers import SubnetPivotAnalyzer
import pandas as pd

analyzer = SubnetPivotAnalyzer()
if not analyzer.connect():
    raise SystemExit('Failed to connect to Neo4j')

try:
    with analyzer.driver.session(database=analyzer.database) as session:
        result = session.run(
            """
            MATCH path = (a:IP)-[r1:CONNECTS]->(b:IP)-[r2:CONNECTS]->(c:IP)-[r3:CONNECTS]->(d:IP)
            WHERE r1.is_attack = 1 AND r2.is_attack = 1 AND r3.is_attack = 1
              AND r2.timestamp > r1.timestamp
              AND r3.timestamp > r2.timestamp
              AND a <> c AND b <> d AND a <> d
            RETURN 
                a.address as hop1_ip,
                b.address as hop2_ip,
                c.address as hop3_ip,
                d.address as hop4_ip,
                a.subnet as hop1_subnet,
                b.subnet as hop2_subnet,
                c.subnet as hop3_subnet,
                d.subnet as hop4_subnet,
                (r2.timestamp - r1.timestamp) / 3600.0 as hours_to_hop2,
                (r3.timestamp - r2.timestamp) / 3600.0 as hours_to_hop3,
                r1.tactic as tactic1,
                r2.tactic as tactic2,
                r3.tactic as tactic3
            """
        ).data()

    df = pd.DataFrame(result)
    print('rows:', len(df))
    print('columns:', df.columns.tolist())
    print(df.head())
    print('unique hop1_ip count', df['hop1_ip'].nunique())
    print('unique hop1_subnet count', df['hop1_subnet'].nunique())
finally:
    analyzer.close()